# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In this section, we'll enumerate the available record sets, their `@id`, and corresponding fields and columns. All references use the entity's `@id`.

In [ ]:
# List available record sets, their @id, and the fields in each
print("Available Record Sets:")
for record_set in dataset.record_sets:
    print(f"- RecordSet name: {record_set.name}, @id: {record_set.id_}")
    print("  Fields and Columns:")
    for field in record_set.fields:
        print(f"    - Field name: {getattr(field, 'name', None)}, @id: {field.id_}, column_id: {getattr(field, 'column_id', None)}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

We'll use the record set and field `@id`s gathered from the previous cell.

In [ ]:
# Extract data from each record set by @id (entity identifier)
# Gather all record set @id values
record_sets_ids = [rs.id_ for rs in dataset.record_sets]
dataframes = {}

# Load all record sets into DataFrames
for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns for RecordSet {record_set_id}:")
    print(df.columns.tolist())
    print("\nSample records:")
    print(df.head(), "\n")

# For demonstration, pick the first available record set
if record_sets_ids:
    selected_record_set_id = record_sets_ids[0]
    selected_df = dataframes[selected_record_set_id]
    print(f"Selected for EDA: {selected_record_set_id}")
else:
    selected_record_set_id = None
    selected_df = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section may include removing outliers, transforming distributions, or grouping data to prepare for further analysis.
> **Note:** We use entity `@id` values (`field.id_`) when referencing fields.

In [ ]:
import numpy as np

if selected_df is not None and not selected_df.empty:
    # Find numeric fields by inspecting dtypes or field metadata
    # We'll use the field types from croissant schema if possible; fall back to pandas autodetection
    # For demonstration, try to find the first numeric column in the DataFrame
    numeric_candidates = selected_df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_candidates:
        # Try to convert columns to numeric if possible
        numeric_candidates = []
        for col in selected_df.columns:
            try:
                selected_df[col] = pd.to_numeric(selected_df[col])
                if pd.api.types.is_numeric_dtype(selected_df[col]):
                    numeric_candidates.append(col)
            except Exception:
                continue

    if numeric_candidates:
        numeric_field = numeric_candidates[0]  # Use first numeric field for example
        threshold = selected_df[numeric_field].mean() if not np.isnan(selected_df[numeric_field].mean()) else 0
        print(f"Using numeric field '@id': {numeric_field} and threshold: {threshold}")
        filtered_df = selected_df[selected_df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())
        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"\nNormalized '{numeric_field}' for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a non-numeric field
        group_candidates = [col for col in selected_df.columns if col != numeric_field and selected_df[col].nunique() < 10]
        group_field = group_candidates[0] if group_candidates else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"\nGrouped data by '{group_field}':")
            print(grouped_df.head())
    else:
        print("No numeric columns detected for EDA.")
else:
    print("No records available for EDA in the selected RecordSet.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

if selected_df is not None and not selected_df.empty and 'numeric_field' in locals():
    plt.figure(figsize=(8, 5))
    selected_df[numeric_field].hist(bins=30)
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.title(f"Distribution of {numeric_field} in {selected_record_set_id}")
    plt.show()

    if 'group_field' in locals() and group_field:
        means = selected_df.groupby(group_field)[numeric_field].mean()
        means.plot(kind='bar', figsize=(8,4))
        plt.ylabel(f"Mean {numeric_field}")
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.show()

## 6. Conclusion
In this notebook, we loaded a mass spectrometry-based extracellular vesicle protein dataset from the Croissant schema, explored its structure via `mlcroissant`, extracted the data with reference to `@id` fields, performed initial filtering and normalization, and visualized the results. 

This approach provides a reproducible, schema-driven path for FAIR machine learning dataset exploration.